
# Piezo Patch Geometry + Actuation Optimizer Starter Notebook

This notebook is a starter framework for optimizing piezo patch placement using the `FE3.py` Euler--Bernoulli finite-element model and the frequency-domain FRF idea from `FE_helpers.py`.

The intended workflow is:

1. Choose a fixed number of piezo patches, `Np`.
2. Parameterize the geometry using patch lengths and gaps.
3. Build a `GeometrySpec` using `build_geometry_arbitrary_piezos(...)`.
4. Assemble the Euler--Bernoulli FE model with `PiezoBeamFE`.
5. For each target mode, search frequency and phase/sign actuation to find the best response for that mode.
6. Combine the modal responses into one multi-mode objective.
7. Repeat for different `Np` values.

The first version uses a direct mechanical FRF solve:

\[
\hat{u}(\omega)=\left(K+i\omega C-\omega^2M\right)^{-1}\Gamma \hat{v}.
\]

This is equivalent to the linear frequency-domain approach in `FE_helpers.py`, but it directly computes the contribution from each patch so that phase/sign optimization is fast.



## 0. Import setup

Run this notebook from your project root if possible, i.e., the directory above `Modeling/`. If the imports fail, update `PROJECT_ROOT` below.


In [2]:

from __future__ import annotations

import copy
import itertools
import math
import sys
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Tuple

import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import differential_evolution

# ---------------------------------------------------------------------
# EDIT THIS if the imports below fail.
# Typical value: PROJECT_ROOT = Path("/path/to/your/project/root")
# ---------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parents[2]

for p in [PROJECT_ROOT, PROJECT_ROOT.parent, Path("/mnt/data")]:
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

# Preferred imports if your project is organized as Modeling/models/...
try:
    from Modeling.models.FE3 import PiezoBeamFE, build_geometry_arbitrary_piezos
    from Modeling.models.beam_properties import PiezoBeamParams
    print("Imported from Modeling.models.*")
except Exception as e1:
    print("Could not import from Modeling.models.*")
    print("Reason:", repr(e1))
    # Fallback imports if FE3.py and beam_properties.py are in the current folder.
    # Note: the uploaded FE3.py itself imports Modeling.models.beam_properties,
    # so your local project-root import is usually the cleanest path.
    try:
        from FE3 import PiezoBeamFE, build_geometry_arbitrary_piezos
        from beam_properties import PiezoBeamParams
        print("Imported from local FE3.py and beam_properties.py")
    except Exception as e2:
        raise ImportError(
            "Could not import FE3/PiezoBeamParams. Run this notebook from the project root "
            "or update PROJECT_ROOT. Original errors:\n"
            f"Modeling import: {repr(e1)}\nLocal import: {repr(e2)}"
        )


Imported from Modeling.models.*


In [2]:
print(Path.cwd())

c:\Users\dlie9\OneDrive - Georgia Institute of Technology\General - Swimmers\Software\Arduino\Arduino Playground\FishCode\multi_mfc_control\Modeling\tasks\Fish



## 1. Design/constraint configuration

This section is intentionally explicit so you can quickly add constraints for patch number, patch lengths, gap sizes, total patch material, etc.

The geometry parameterization is:

\[
[g_0,\ell_1,g_1,\ell_2,g_2,\ldots,g_{N_p-1},\ell_{N_p}],
\]

where:

- `g0` is the root gap before patch 1,
- `gj` for `j >= 1` is the gap before patch `j+1`,
- `ell_j` is the length of patch `j`,
- the final tip gap is computed as whatever beam length remains.

The optimizer variables are stored as:

```text
z = [ell_1, ell_2, ..., ell_Np, gap_before_1, gap_before_2, ..., gap_before_Np]
```

where `gap_before_1` is the root gap.


In [3]:

@dataclass
class GeometryConstraints:
    """Constraints and bounds for the patch-layout optimizer."""

    # Beam length used for the optimizer [m].
    # If None, the notebook uses base_params.L_b.
    L: Optional[float] = None

    # Patch counts to test. We usually optimize each Np separately.
    n_patch_candidates: Tuple[int, ...] = (1, 2, 3, 4)

    # Patch length bounds [m]. Can be scalar tuple or per-patch dictionary via fixed_lengths.
    patch_length_bounds: Tuple[float, float] = (5e-3, 80e-3)

    # Gap before first patch/root gap bounds [m].
    root_gap_bounds: Tuple[float, float] = (0.0, 150e-3)

    # Gaps between patches [m].
    internal_gap_bounds: Tuple[float, float] = (0.5e-3, 150e-3)

    # Final tip gap is not an optimization variable; it is computed from remaining length.
    tip_gap_bounds: Tuple[float, float] = (0.0, 150e-3)

    # Optional total active patch length constraint [m]. Use None to disable.
    total_patch_length_bounds: Optional[Tuple[float, float]] = None

    # Optional fixed patch lengths: {patch_index_0_based: fixed_length_m}
    # Example: {0: 30e-3} fixes patch 1 length to 30 mm.
    fixed_lengths: Dict[int, float] = field(default_factory=dict)

    # Optional fixed gaps before each patch: {gap_index_0_based: fixed_gap_m}
    # gap_index=0 means root gap before patch 1.
    # gap_index=1 means gap between patch 1 and patch 2.
    fixed_gaps_before: Dict[int, float] = field(default_factory=dict)

    # Penalty scales. Increase if the optimizer accepts invalid layouts.
    invalid_penalty: float = 1e12
    soft_penalty: float = 1e9


@dataclass
class MeshConfig:
    """Element sizes used when building the FE geometry."""
    h_patch: float = 1.0e-3
    h_gap: float = 1.0e-3


@dataclass
class ModeSpec:
    """One mode-specific objective term."""
    name: str
    mode_index: int                 # 1-based mode index
    weight: float = 1.0

    # If None, use a window around the FE eigenfrequency for this mode.
    freq_window_hz: Optional[Tuple[float, float]] = None

    # Used only when freq_window_hz is None.
    rel_window: float = 0.25        # +/- 25% around eigenfrequency
    min_window_hz: float = 0.25     # avoid a zero-width/too-small window

    # Frequency samples for the inner FRF search.
    n_freq: int = 120

    # Optional baseline for normalized score. If None, uses raw response.
    baseline_response: Optional[float] = None


@dataclass
class ActuationConfig:
    """Mode-specific actuation search settings."""
    voltage_amplitude: float = 1.0  # Use 1 V for per-volt FRF; scale later.

    # 'binary' = each patch has sign +/-1.
    # 'continuous' = each patch can use any phase angle.
    phase_type: str = "binary"

    # Brute-force binary signs up to this patch count. Above this, use greedy sign flips.
    max_bruteforce_patches: int = 12


@dataclass
class ObjectiveConfig:
    """Top-level objective settings."""
    output_metric: str = "tip"      # starter version supports 'tip'
    use_modal_damping: bool = True
    use_rayleigh_damping: bool = True
    return_negative_for_minimizer: bool = True


# ---------------------------------------------------------------------
# USER EDIT SECTION
# ---------------------------------------------------------------------
base_params = PiezoBeamParams()

geom_constraints = GeometryConstraints(
    L=0.365,  # None => base_params.L_b. Set explicitly if needed, e.g. 0.3185.
    n_patch_candidates=(1, 2, 3, 4),
    patch_length_bounds=(5e-3, 80e-3),
    root_gap_bounds=(0.0, 150e-3),
    internal_gap_bounds=(3e-3, 150e-3),
    tip_gap_bounds=(0.0, 150e-3),
    total_patch_length_bounds=None,  # e.g. (20e-3, 160e-3)
    fixed_lengths={},               # e.g. {0: 30e-3}
    fixed_gaps_before={},           # e.g. {0: 5e-3}
)

mesh_config = MeshConfig(
    h_patch=1.0e-3,
    h_gap=1.0e-3,
)

# Recommended: start with eigenfrequency-centered windows.
# Later, replace with fixed windows such as (0.5, 3.0), (2.0, 6.0), (5.0, 10.0).
mode_specs = [
    ModeSpec(name="mode_1", mode_index=1, weight=1.0, freq_window_hz=None, rel_window=0.30, n_freq=120),
    ModeSpec(name="mode_2", mode_index=2, weight=1.0, freq_window_hz=None, rel_window=0.30, n_freq=120),
    ModeSpec(name="mode_3", mode_index=3, weight=1.0, freq_window_hz=None, rel_window=0.30, n_freq=120),
]

actuation_config = ActuationConfig(
    voltage_amplitude=1.0,
    phase_type="binary",  # use 'continuous' if arbitrary phase per patch is allowed
    max_bruteforce_patches=12,
)

objective_config = ObjectiveConfig(
    output_metric="tip",
    use_modal_damping=True,
    use_rayleigh_damping=True,
)

# Effective beam length for optimization.
L_OPT = base_params.L_b if geom_constraints.L is None else geom_constraints.L
print(f"Optimization beam length L_OPT = {L_OPT:.6f} m")
print(f"Default base_params has {base_params.S} patches, but optimizer will override geometry.")


Optimization beam length L_OPT = 0.365000 m
Default base_params has 31 patches, but optimizer will override geometry.



## 2. Geometry encoding/decoding and constraint penalties

The optimizer will run separately for each fixed patch count `Np`. This avoids the awkward issue that changing the number of patches changes the dimension of the design vector.


In [4]:

def _fixed_bound(value: float, eps: float = 0.0) -> Tuple[float, float]:
    """Return a fixed or nearly fixed bound. eps=0 usually works with scipy DE."""
    return (value - eps, value + eps)


def make_bounds_for_Np(Np: int, gc: GeometryConstraints) -> List[Tuple[float, float]]:
    """
    Bounds for z = [lengths(0:Np), gaps_before(0:Np)].
    gaps_before[0] is the root gap; gaps_before[j>0] is the gap before patch j.
    """
    bounds: List[Tuple[float, float]] = []

    # Patch lengths
    for j in range(Np):
        if j in gc.fixed_lengths:
            bounds.append(_fixed_bound(gc.fixed_lengths[j]))
        else:
            bounds.append(gc.patch_length_bounds)

    # Gaps before each patch
    for j in range(Np):
        if j in gc.fixed_gaps_before:
            bounds.append(_fixed_bound(gc.fixed_gaps_before[j]))
        elif j == 0:
            bounds.append(gc.root_gap_bounds)
        else:
            bounds.append(gc.internal_gap_bounds)

    return bounds


def decode_design(z: np.ndarray, Np: int, gc: GeometryConstraints, L: float) -> dict:
    """Convert optimizer vector z into xL/xR arrays and derived gap information."""
    z = np.asarray(z, dtype=float)
    if z.size != 2*Np:
        raise ValueError(f"Expected z of length {2*Np}, got {z.size}.")

    lengths = z[:Np].copy()
    gaps_before = z[Np:].copy()

    xL = np.zeros(Np)
    xR = np.zeros(Np)

    x = gaps_before[0]
    for j in range(Np):
        xL[j] = x
        xR[j] = x + lengths[j]
        if j < Np - 1:
            x = xR[j] + gaps_before[j+1]

    tip_gap = L - xR[-1]

    return {
        "Np": Np,
        "lengths": lengths,
        "gaps_before": gaps_before,
        "xL": xL,
        "xR": xR,
        "tip_gap": tip_gap,
        "total_patch_length": float(np.sum(lengths)),
        "used_length_to_last_patch": float(xR[-1]),
    }


def constraint_penalty(layout: dict, gc: GeometryConstraints, L: float) -> float:
    """Soft/hard penalties for geometry constraints not fully enforced by bounds."""
    penalty = 0.0
    Np = layout["Np"]
    xL = layout["xL"]
    xR = layout["xR"]
    lengths = layout["lengths"]
    gaps_before = layout["gaps_before"]
    tip_gap = layout["tip_gap"]

    # Hard invalid checks.
    if np.any(~np.isfinite(xL)) or np.any(~np.isfinite(xR)):
        return gc.invalid_penalty
    if np.any(lengths <= 0):
        return gc.invalid_penalty
    if np.any(xL < -1e-12) or np.any(xR > L + 1e-12):
        return gc.invalid_penalty
    if np.any(xL >= xR):
        return gc.invalid_penalty
    if Np > 1 and np.any(xR[:-1] > xL[1:] + 1e-12):
        return gc.invalid_penalty

    # Tip gap bounds are not direct optimizer bounds, so enforce here.
    tip_min, tip_max = gc.tip_gap_bounds
    if tip_gap < tip_min:
        penalty += gc.soft_penalty * (tip_min - tip_gap)**2
    if tip_gap > tip_max:
        penalty += gc.soft_penalty * (tip_gap - tip_max)**2

    # Total active material constraint.
    if gc.total_patch_length_bounds is not None:
        lo, hi = gc.total_patch_length_bounds
        total_len = layout["total_patch_length"]
        if total_len < lo:
            penalty += gc.soft_penalty * (lo - total_len)**2
        if total_len > hi:
            penalty += gc.soft_penalty * (total_len - hi)**2

    return float(penalty)


# Quick smoke test for one design vector.
Np_test = 3
bounds_test = make_bounds_for_Np(Np_test, geom_constraints)
z_mid = np.array([(lo + hi)/2 for lo, hi in bounds_test])
layout_test = decode_design(z_mid, Np_test, geom_constraints, L_OPT)
print(layout_test)
print("penalty:", constraint_penalty(layout_test, geom_constraints, L_OPT))


{'Np': 3, 'lengths': array([0.0425, 0.0425, 0.0425]), 'gaps_before': array([0.075 , 0.0765, 0.0765]), 'xL': array([0.075, 0.194, 0.313]), 'xR': array([0.1175, 0.2365, 0.3555]), 'tip_gap': 0.009500000000000008, 'total_patch_length': 0.1275, 'used_length_to_last_patch': 0.3555}
penalty: 0.0


In [7]:
bounds_test
z_mid
layout_test

{'Np': 3,
 'lengths': array([0.0425, 0.0425, 0.0425]),
 'gaps_before': array([0.075 , 0.0765, 0.0765]),
 'xL': array([0.075, 0.194, 0.313]),
 'xR': array([0.1175, 0.2365, 0.3555]),
 'tip_gap': 0.009500000000000008,
 'total_patch_length': 0.1275,
 'used_length_to_last_patch': 0.3555}


## 3. Build FE model from a candidate layout

This function creates a fresh `GeometrySpec` and `PiezoBeamFE` model for each candidate. This is the simplest and most robust way to support changing patch lengths/gaps/locations.

For now, patch width/thickness/material are inherited from `PiezoBeamParams`. If you later want to optimize individual patch widths or thicknesses, the cross-section formulas need to be generalized.


In [15]:

def make_geometry_from_layout(layout: dict, params: PiezoBeamParams, mesh_cfg: MeshConfig, L: float):
    """Create a GeometrySpec from xL/xR using the same patch/gap property logic as FE3.py."""
    xL = np.asarray(layout["xL"], dtype=float)
    xR = np.asarray(layout["xR"], dtype=float)

    # Structural properties. These match the logic in FE3.py / geometry_from_params.
    rhoA_patch = params.b * (params.rho_s*params.hs + 2.0*params.rho_p*params.hp)
    rhoA_gap   = params.b * params.rho_s * params.hs
    EI_patch   = params.YI
    EI_gap     = params.YI_s

    geom = build_geometry_arbitrary_piezos(
        L=L,
        xL=xL,
        xR=xR,
        EI_patch=EI_patch,
        rhoA_patch=rhoA_patch,
        EI_gap=EI_gap,
        rhoA_gap=rhoA_gap,
        h_patch=mesh_cfg.h_patch,
        h_gap=mesh_cfg.h_gap,
    )
    return geom


def build_fe_from_layout(layout: dict, base_params: PiezoBeamParams, mesh_cfg: MeshConfig, L: float) -> PiezoBeamFE:
    """Build a PiezoBeamFE instance for one candidate layout."""
    params = copy.deepcopy(base_params)
    params.geometry = make_geometry_from_layout(layout, params, mesh_cfg, L)
    fe = PiezoBeamFE(params)
    return fe


def get_damping_matrix(fe: PiezoBeamFE, base_params: PiezoBeamParams, obj_cfg: ObjectiveConfig) -> np.ndarray:
    """Return mechanical damping matrix for the direct FRF solve."""
    D = np.zeros_like(fe.K_red)

    if obj_cfg.use_modal_damping and hasattr(fe, "C_red"):
        D = D + fe.C_red

    # This mirrors the damping combination used inside FE3.build_ode_system().
    if obj_cfg.use_rayleigh_damping:
        D = D + base_params.c_alpha*fe.M_red + base_params.c_beta*fe.K_red

    return D


def tip_reduced_index(fe: PiezoBeamFE) -> int:
    """Return the reduced DOF index corresponding to full-system tip displacement."""
    n_nodes = len(fe.geom.x_nodes)
    full_tip_w_dof = 2*(n_nodes - 1)
    matches = np.where(fe.free_dofs == full_tip_w_dof)[0]
    if len(matches) != 1:
        raise RuntimeError("Could not locate tip displacement DOF in reduced system.")
    return int(matches[0])


# Smoke test: build one FE model.
fe_test = build_fe_from_layout(layout_test, base_params, mesh_config, L_OPT)
print("N free DOFs:", fe_test.K_red.shape[0])
print("N patches:", fe_test.Gamma_red.shape[1])
print("First five frequencies [Hz]:", fe_test.freq[:5])
print("Tip reduced index:", tip_reduced_index(fe_test))


N free DOFs: 808
N patches: 3
First five frequencies [Hz]: [  2.13280174  14.06627824  42.12288236  88.42105963 152.98918788]
Tip reduced index: 806


In [14]:
layout_test
base_params
mesh_config
L_OPT

0.4


## 4. Direct patch-to-output FRF contribution

For one frequency, compute the complex contribution from each patch to the tip response:

\[
h_j(\omega)=e_{tip}^T\left(K+i\omega C-\omega^2M\right)^{-1}\Gamma_j.
\]

Then for a voltage vector \(\hat{v}\),

\[
\hat{y}=\sum_j h_j\hat{v}_j.
\]

This makes phase/sign optimization cheap.


In [16]:

def patch_to_tip_contributions(fe: PiezoBeamFE, omega: float, D: np.ndarray) -> np.ndarray:
    """
    Return h_j values for unit voltage on each patch at angular frequency omega.

    h has shape (Np,), where y_tip = h @ v_complex.
    """
    K = fe.K_red
    M = fe.M_red
    Gamma = fe.Gamma_red

    Z = K + 1j*omega*D - (omega**2)*M

    # Solve all unit-patch RHS columns at once.
    U_cols = np.linalg.solve(Z, Gamma)  # shape: (Nfree, Np)

    idx_tip = tip_reduced_index(fe)
    h = U_cols[idx_tip, :]
    return h


def best_actuation_for_h(h: np.ndarray, act_cfg: ActuationConfig) -> dict:
    """
    Given complex patch contributions h_j, find best phase/sign vector.

    Returns a dict with response magnitude and complex voltage vector.
    """
    h = np.asarray(h, dtype=complex)
    Np = h.size
    A = act_cfg.voltage_amplitude

    if act_cfg.phase_type.lower() == "continuous":
        # Choose phases so all h_j v_j align on the positive real axis.
        phases = -np.angle(h)
        v = A * np.exp(1j*phases)
        y = np.dot(h, v)
        return {
            "response": float(abs(y)),
            "voltage_vector": v,
            "phases_rad": phases,
            "signs": None,
            "method": "continuous_phase_alignment",
        }

    if act_cfg.phase_type.lower() == "binary":
        if Np <= act_cfg.max_bruteforce_patches:
            sign_array = np.array(list(itertools.product([-1.0, 1.0], repeat=Np)))
            y_all = sign_array @ h
            k_best = int(np.argmax(np.abs(y_all)))
            signs = sign_array[k_best]
            v = A * signs.astype(complex)
            return {
                "response": float(abs(y_all[k_best]) * A),
                "voltage_vector": v,
                "phases_rad": np.where(signs > 0, 0.0, np.pi),
                "signs": signs,
                "method": "binary_bruteforce",
            }
        else:
            # Greedy coordinate sign-flip fallback for many patches.
            signs = np.ones(Np)
            current = abs(np.dot(h, signs))
            improved = True
            while improved:
                improved = False
                for j in range(Np):
                    trial = signs.copy()
                    trial[j] *= -1.0
                    val = abs(np.dot(h, trial))
                    if val > current:
                        signs = trial
                        current = val
                        improved = True
            v = A * signs.astype(complex)
            return {
                "response": float(current * A),
                "voltage_vector": v,
                "phases_rad": np.where(signs > 0, 0.0, np.pi),
                "signs": signs,
                "method": "binary_greedy",
            }

    raise ValueError(f"Unknown phase_type={act_cfg.phase_type!r}. Use 'binary' or 'continuous'.")


# Smoke test at mode 1 frequency.
D_test = get_damping_matrix(fe_test, base_params, objective_config)
w0 = 2*np.pi*fe_test.freq[0]
h_test = patch_to_tip_contributions(fe_test, w0, D_test)
best_test = best_actuation_for_h(h_test, actuation_config)
print("h_test:", h_test)
print("best response:", best_test["response"])
print("best voltage vector:", best_test["voltage_vector"])


h_test: [ 4.59935643e-08+3.92717889e-04j -3.03568095e-06+1.68017661e-04j
 -2.85226433e-06+1.94219553e-05j]
best response: 0.0005801869177200711
best voltage vector: [-1.+0.j -1.+0.j -1.+0.j]



## 5. Mode-specific frequency and actuation search

For each target mode, this function:

1. chooses a frequency window,
2. sweeps the FRF over that window,
3. finds the best phase/sign vector at every frequency,
4. returns the best frequency and actuation vector for that mode.

This is the inner optimization. The outer optimizer only sees the geometry score.


In [17]:

def frequency_window_for_mode(fe: PiezoBeamFE, mode: ModeSpec) -> Tuple[float, float]:
    """Return [f_min, f_max] for one mode in Hz."""
    if mode.freq_window_hz is not None:
        return mode.freq_window_hz

    idx = mode.mode_index - 1
    if idx < 0 or idx >= len(fe.freq):
        raise ValueError(f"mode_index={mode.mode_index} is out of range for computed modes.")

    fc = float(fe.freq[idx])
    half_width = max(mode.rel_window * fc, mode.min_window_hz)
    f_min = max(1e-6, fc - half_width)
    f_max = fc + half_width
    return f_min, f_max


def evaluate_one_mode(
    fe: PiezoBeamFE,
    mode: ModeSpec,
    act_cfg: ActuationConfig,
    obj_cfg: ObjectiveConfig,
    base_params: PiezoBeamParams,
) -> dict:
    """Evaluate best response for one mode and one FE geometry."""
    if obj_cfg.output_metric.lower() != "tip":
        raise NotImplementedError("Starter notebook currently supports output_metric='tip'.")

    D = get_damping_matrix(fe, base_params, obj_cfg)
    f_min, f_max = frequency_window_for_mode(fe, mode)
    freqs = np.linspace(f_min, f_max, mode.n_freq)

    best = None
    responses = np.zeros_like(freqs, dtype=float)

    for k, f_hz in enumerate(freqs):
        omega = 2*np.pi*f_hz
        h = patch_to_tip_contributions(fe, omega, D)
        act = best_actuation_for_h(h, act_cfg)
        responses[k] = act["response"]

        if best is None or act["response"] > best["response"]:
            best = {
                **act,
                "freq_hz": float(f_hz),
                "omega": float(omega),
                "h": h,
            }

    raw_response = best["response"]
    if mode.baseline_response is None:
        normalized_response = raw_response
    else:
        normalized_response = raw_response / mode.baseline_response

    return {
        "mode_name": mode.name,
        "mode_index": mode.mode_index,
        "weight": mode.weight,
        "natural_freq_hz": float(fe.freq[mode.mode_index - 1]),
        "freq_window_hz": (float(f_min), float(f_max)),
        "freq_grid_hz": freqs,
        "response_grid": responses,
        "best": best,
        "raw_response": float(raw_response),
        "normalized_response": float(normalized_response),
        "weighted_score": float(mode.weight * normalized_response),
    }


def evaluate_layout(
    layout: dict,
    base_params: PiezoBeamParams,
    mesh_cfg: MeshConfig,
    gc: GeometryConstraints,
    modes: List[ModeSpec],
    act_cfg: ActuationConfig,
    obj_cfg: ObjectiveConfig,
    L: float,
    return_fe: bool = False,
) -> dict:
    """Build FE model and evaluate all requested mode objectives."""
    penalty = constraint_penalty(layout, gc, L)
    if penalty >= gc.invalid_penalty:
        return {
            "valid": False,
            "J_raw": -np.inf,
            "penalty": penalty,
            "J_penalized": -np.inf,
            "layout": layout,
            "mode_results": [],
            "fe": None,
        }

    try:
        fe = build_fe_from_layout(layout, base_params, mesh_cfg, L)
        mode_results = [
            evaluate_one_mode(fe, mode, act_cfg, obj_cfg, base_params)
            for mode in modes
        ]
        J_raw = float(sum(r["weighted_score"] for r in mode_results))
        J_penalized = J_raw - penalty
        out = {
            "valid": True,
            "J_raw": J_raw,
            "penalty": penalty,
            "J_penalized": J_penalized,
            "layout": layout,
            "mode_results": mode_results,
        }
        if return_fe:
            out["fe"] = fe
        return out
    except Exception as e:
        return {
            "valid": False,
            "J_raw": -np.inf,
            "penalty": gc.invalid_penalty,
            "J_penalized": -np.inf,
            "layout": layout,
            "mode_results": [],
            "error": repr(e),
            "fe": None,
        }


# Smoke test full evaluation.
eval_test = evaluate_layout(
    layout_test,
    base_params,
    mesh_config,
    geom_constraints,
    mode_specs,
    actuation_config,
    objective_config,
    L_OPT,
    return_fe=False,
)
print("valid:", eval_test["valid"])
print("J_raw:", eval_test["J_raw"])
for r in eval_test["mode_results"]:
    print(r["mode_name"], "fn=", r["natural_freq_hz"], "best f=", r["best"]["freq_hz"], "resp=", r["raw_response"], "v=", r["best"]["voltage_vector"])


valid: True
J_raw: 0.0007168565488804947
mode_1 fn= 2.132801738763765 best f= 2.127424927657638 resp= 0.0005779968815596611 v= [-1.+0.j -1.+0.j -1.+0.j]
mode_2 fn= 14.066278239966934 best f= 14.03081703431996 resp= 9.71613587699211e-05 v= [-1.+0.j -1.+0.j -1.+0.j]
mode_3 fn= 42.122882360163 best f= 42.01669021975923 resp= 4.1698308550912515e-05 v= [-1.+0.j  1.+0.j  1.+0.j]



## 6. Objective wrapper for SciPy

`scipy.optimize.differential_evolution` minimizes, so we return `-J` for valid designs.


In [18]:

def objective_z(
    z: np.ndarray,
    Np: int,
    base_params: PiezoBeamParams,
    mesh_cfg: MeshConfig,
    gc: GeometryConstraints,
    modes: List[ModeSpec],
    act_cfg: ActuationConfig,
    obj_cfg: ObjectiveConfig,
    L: float,
) -> float:
    layout = decode_design(z, Np, gc, L)
    result = evaluate_layout(
        layout=layout,
        base_params=base_params,
        mesh_cfg=mesh_cfg,
        gc=gc,
        modes=modes,
        act_cfg=act_cfg,
        obj_cfg=obj_cfg,
        L=L,
        return_fe=False,
    )

    if not result["valid"]:
        return gc.invalid_penalty

    J = result["J_penalized"]
    if obj_cfg.return_negative_for_minimizer:
        return -J
    return J


def optimize_for_Np(
    Np: int,
    base_params: PiezoBeamParams,
    mesh_cfg: MeshConfig,
    gc: GeometryConstraints,
    modes: List[ModeSpec],
    act_cfg: ActuationConfig,
    obj_cfg: ObjectiveConfig,
    L: float,
    maxiter: int = 30,
    popsize: int = 10,
    seed: int = 1,
    polish: bool = True,
    workers: int = 1,
) -> dict:
    """Run differential evolution for one fixed patch count."""
    bounds = make_bounds_for_Np(Np, gc)

    result = differential_evolution(
        func=lambda z: objective_z(z, Np, base_params, mesh_cfg, gc, modes, act_cfg, obj_cfg, L),
        bounds=bounds,
        maxiter=maxiter,
        popsize=popsize,
        seed=seed,
        polish=polish,
        updating="immediate" if workers == 1 else "deferred",
        workers=workers,
        tol=1e-4,
        disp=True,
    )

    best_layout = decode_design(result.x, Np, gc, L)
    best_eval = evaluate_layout(
        best_layout,
        base_params,
        mesh_cfg,
        gc,
        modes,
        act_cfg,
        obj_cfg,
        L,
        return_fe=True,
    )

    return {
        "Np": Np,
        "scipy_result": result,
        "best_layout": best_layout,
        "best_eval": best_eval,
    }


def print_optimization_summary(opt_result: dict) -> None:
    """Human-readable summary of one optimization result."""
    Np = opt_result["Np"]
    layout = opt_result["best_layout"]
    ev = opt_result["best_eval"]

    print("="*80)
    print(f"Np = {Np}")
    print(f"J_raw       = {ev['J_raw']:.6e}")
    print(f"penalty     = {ev['penalty']:.6e}")
    print(f"J_penalized = {ev['J_penalized']:.6e}")
    print("xL [mm]     =", np.round(1e3*layout["xL"], 3))
    print("xR [mm]     =", np.round(1e3*layout["xR"], 3))
    print("lengths [mm]=", np.round(1e3*layout["lengths"], 3))
    print("gaps before [mm]=", np.round(1e3*layout["gaps_before"], 3))
    print("tip gap [mm]=", round(1e3*layout["tip_gap"], 3))
    print("total patch length [mm]=", round(1e3*layout["total_patch_length"], 3))
    print("Mode results:")
    for r in ev["mode_results"]:
        b = r["best"]
        print(
            f"  {r['mode_name']}: fn={r['natural_freq_hz']:.4f} Hz, "
            f"best f={b['freq_hz']:.4f} Hz, resp={r['raw_response']:.6e}, "
            f"method={b['method']}"
        )
        if b.get("signs") is not None:
            print("    signs:", b["signs"].astype(int))
        print("    phases [deg]:", np.round(np.rad2deg(b["phases_rad"]), 1))
    print("="*80)



## 7. Single-mode runs first: recommended diagnostic step

Before jumping into multi-mode optimization, run individual mode optimizations. This is useful because it tells you:

- where each mode wants patch edges,
- whether the frequency windows make sense,
- whether the phase/sign patterns agree with your modal intuition,
- whether different modes are fighting each other geometrically.

The final design should still be optimized as a multi-mode objective, but single-mode optima are excellent diagnostics and can provide candidate initial layouts.


In [21]:

# ---------------------------------------------------------------------
# Optional: run individual mode optimizations.
# Start with small maxiter/popsize to verify the pipeline, then increase.
# ---------------------------------------------------------------------
RUN_SINGLE_MODE_DEMO = True

single_mode_results = []
if RUN_SINGLE_MODE_DEMO:
    for mode in mode_specs:
        print(f"\nOptimizing {mode.name} alone...")
        modes_one = [mode]
        res = optimize_for_Np(
            Np=3,
            base_params=base_params,
            mesh_cfg=mesh_config,
            gc=geom_constraints,
            modes=modes_one,
            act_cfg=actuation_config,
            obj_cfg=objective_config,
            L=L_OPT,
            maxiter=15,
            popsize=8,
            seed=10 + mode.mode_index,
            polish=True,
            workers=1,
        )
        print_optimization_summary(res)
        single_mode_results.append(res)
else:
    print("Set RUN_SINGLE_MODE_DEMO=True to run single-mode diagnostic optimizations.")



Optimizing mode_1 alone...
differential_evolution step 1: f(x)= -0.001393464459241388
differential_evolution step 2: f(x)= -0.0014324169774277318
differential_evolution step 3: f(x)= -0.0014478117249446304
differential_evolution step 4: f(x)= -0.0016157519453311449
differential_evolution step 5: f(x)= -0.0016157519453311449
differential_evolution step 6: f(x)= -0.0016450820525997641
differential_evolution step 7: f(x)= -0.0016450820525997641
differential_evolution step 8: f(x)= -0.0017399893681351534
differential_evolution step 9: f(x)= -0.0017774146475573152
differential_evolution step 10: f(x)= -0.0017938909293901111
differential_evolution step 11: f(x)= -0.0017938909293901111
differential_evolution step 12: f(x)= -0.0018267030954170877
differential_evolution step 13: f(x)= -0.0018385655683109032
differential_evolution step 14: f(x)= -0.0018385655683109032
differential_evolution step 15: f(x)= -0.0018385655683109032
Polishing solution with 'L-BFGS-B'


c:\Users\dlie9\OneDrive - Georgia Institute of Technology\General - Swimmers\Software\Arduino\Arduino Playground\FishCode\multi_mfc_control\Modeling\models\FE3.py:220: RuntimeWarning: invalid value encountered in sqrt
  omega = np.sqrt(eigvals)


Np = 3
J_raw       = 1.838566e-03
penalty     = 0.000000e+00
J_penalized = 1.838566e-03
xL [mm]     = [  2.455  81.414 222.558]
xR [mm]     = [ 80.782 149.34  279.142]
lengths [mm]= [78.327 67.926 56.584]
gaps before [mm]= [ 2.455  0.632 73.219]
tip gap [mm]= 120.858
total patch length [mm]= 202.837
Mode results:
  mode_1: fn=4.3928 Hz, best f=4.3818 Hz, resp=1.838566e-03, method=binary_bruteforce
    signs: [-1 -1 -1]
    phases [deg]: [180. 180. 180.]

Optimizing mode_2 alone...
differential_evolution step 1: f(x)= -0.0003478352579174488
differential_evolution step 2: f(x)= -0.0003478352579174488
differential_evolution step 3: f(x)= -0.0003547180401747474
differential_evolution step 4: f(x)= -0.0003713450424399925
differential_evolution step 5: f(x)= -0.0003713450424399925
differential_evolution step 6: f(x)= -0.0003765908182388327
differential_evolution step 7: f(x)= -0.0004154180164837038
differential_evolution step 8: f(x)= -0.0004154180164837038
differential_evolution step 9: f(x


## 8. Multi-mode optimization

This is the main design problem: one geometry, but each mode gets its own best frequency and actuation phase/sign vector.


In [ ]:

# ---------------------------------------------------------------------
# Main multi-mode optimization.
# Start small. Increase maxiter/popsize after the smoke tests work.
# ---------------------------------------------------------------------
RUN_MULTIMODE_DEMO = False

multi_results = []
if RUN_MULTIMODE_DEMO:
    for Np in geom_constraints.n_patch_candidates:
        print(f"\nOptimizing multi-mode design for Np={Np}...")
        res = optimize_for_Np(
            Np=Np,
            base_params=base_params,
            mesh_cfg=mesh_config,
            gc=geom_constraints,
            modes=mode_specs,
            act_cfg=actuation_config,
            obj_cfg=objective_config,
            L=L_OPT,
            maxiter=25,
            popsize=8,
            seed=100 + Np,
            polish=True,
            workers=1,  # set >1 after confirming everything is pickle-safe
        )
        print_optimization_summary(res)
        multi_results.append(res)

    # Sort from best to worst.
    multi_results = sorted(multi_results, key=lambda r: r["best_eval"]["J_penalized"], reverse=True)
    print("\nBest overall:")
    print_optimization_summary(multi_results[0])
else:
    print("Set RUN_MULTIMODE_DEMO=True to run the full multi-mode optimization.")



Optimizing multi-mode design for Np=1...


KeyboardInterrupt: 


## 9. Plot helpers

These functions help inspect one optimized layout and its per-mode FRF curves.


In [ ]:

def plot_layout(layout: dict, L: float, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(9, 1.8))
    else:
        fig = ax.figure

    ax.plot([0, L], [0, 0], linewidth=3)
    for j, (a, b) in enumerate(zip(layout["xL"], layout["xR"])):
        ax.fill_between([a, b], [-0.08, -0.08], [0.08, 0.08], alpha=0.35)
        ax.text((a+b)/2, 0.12, f"P{j+1}", ha="center", va="bottom")

    ax.set_xlim(-0.02*L, 1.02*L)
    ax.set_ylim(-0.25, 0.35)
    ax.set_yticks([])
    ax.set_xlabel("x [m]")
    ax.set_title("Patch layout")
    ax.grid(True, axis="x", alpha=0.3)
    return fig, ax


def plot_mode_response_grids(eval_result: dict):
    for r in eval_result["mode_results"]:
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(r["freq_grid_hz"], r["response_grid"])
        ax.axvline(r["natural_freq_hz"], linestyle="--", label="natural freq")
        ax.axvline(r["best"]["freq_hz"], linestyle=":", label="best FRF freq")
        ax.set_xlabel("Frequency [Hz]")
        ax.set_ylabel("Best tip response per volt [m/V]")
        ax.set_title(r["mode_name"])
        ax.grid(True, alpha=0.3)
        ax.legend()
        plt.show()


# Example usage after running an optimization:
# best = multi_results[0]
# plot_layout(best["best_layout"], L_OPT)
# plot_mode_response_grids(best["best_eval"])



## 10. Notes and next extensions

Recommended next steps:

1. Validate this FE/FRF evaluator against one or two COMSOL cases.
2. Start with `phase_type='binary'` if your real driver only supports opposite-polarity patches.
3. Use single-mode runs to identify candidate layouts and conflicts.
4. Use multi-mode optimization for the final design.
5. Add line-average/RMS output metric if needed.
6. Add water added mass/damping if the current model overpredicts resonance frequencies compared with submerged COMSOL/experiment.
7. If electrical shunts/free piezos are included, update capacitance for variable patch lengths because `Cp_scalar` in `PiezoBeamParams` is based on the default patch length.
